### Wikipedia Retriever

In [3]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [10]:
query = "India vs pakistan?"

results = retriever.invoke(query)
print("Number of documents retrieved:", len(results))

Number of documents retrieved: 2


In [9]:
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Title: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:500]}...")  # Print the first 500 characters
    print("\n" + "="*80 + "\n")

Result 1:
Title: India–Pakistan cricket rivalry
Content: The India–Pakistan cricket rivalry is one of the most intense sports rivalries in the world. Matches between the teams are considered some of the biggest in the world and are among the most-viewed in all of sports.
The two teams have played a total of 212 times, with Pakistan winning 88 matches and India winning 81. In Tests and ODIs, Pakistan has been victorious in more games than India, while India has won more games in T20Is. In ICC World Cups, the two teams have met head to head in 17 matche...


Result 2:
Title: The Greatest Rivalry: India vs Pakistan
Content: The Greatest Rivalry: India vs Pakistan is a 2025 Indian sports documentary series that premiered globally on Netflix on 7 February 2025. The three-part series traces the cricketing rivalry between the India and Pakistan, interweaving archived match footage, interviews with former players, and commentary on the cultural and political backdrop of the contests.


== Syno

### Vector Store Retriever

In [13]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [14]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
    Document(page_content="RAG combines retrieval and generation for better responses."),
    Document(page_content="Wikipedia is a free online encyclopedia with millions of articles."),
    Document(page_content="India and Pakistan have a long history of cricket rivalry."),
    Document(page_content="The India vs Pakistan cricket match is one of the most anticipated events in the sport."),
    Document(page_content="RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval of relevant documents with generation of responses using language models.")
]

In [17]:
embedding_model = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = "my_collection"
)

In [18]:
# There are two ways to retrieve the similar document as query from the vecorstore.

query = "What is RAG?"

# 1. Using the retriever interface
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
results = retriever.invoke(query)
print("Results using retriever interface:")
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}")

# 2. Using the similarity search interface
results = vectorstore.similarity_search(query, k=2)
print("\nResults using similarity search interface:")
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}")

Results using retriever interface:
Result 1: RAG combines retrieval and generation for better responses.
Result 2: RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval of relevant documents with generation of responses using language models.

Results using similarity search interface:
Result 1: RAG combines retrieval and generation for better responses.
Result 2: RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval of relevant documents with generation of responses using language models.


### Maximal Marginal Relevance (MMR)

* MMR can pick the results that are not only relevant to the query but also different from each other.

In [19]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
    Document(page_content="RAG combines retrieval and generation for better responses."),
    Document(page_content="Wikipedia is a free online encyclopedia with millions of articles."),
    Document(page_content="India and Pakistan have a long history of cricket rivalry."),
    Document(page_content="The India vs Pakistan cricket match is one of the most anticipated events in the sport."),
    Document(page_content="RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval of relevant documents with generation of responses using language models.")
]

In [22]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents = documents,
    embedding = embedding_model
)

In [ ]:
query = "What is RAG?"

retriever_1 = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "lambda_mult": 1} # k is the number of documents to return, lambda_mult is the diversity factor (0.5 means equal weight to relevance and diversity.
)

retriever_2 = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "lambda_mult": 0.5} 
)

results_1 = retriever_1.invoke(query)
results_2 = retriever_2.invoke(query)

print("Results with lambda_mult=1 (more relevance):")
for i, doc in enumerate(results_1):
    print(f"Result {i+1}: {doc.page_content}")

print("\nResults with lambda_mult=0.5 (more diversity):")
for i, doc in enumerate(results_2):
    print(f"Result {i+1}: {doc.page_content}")

Results with lambda_mult=1 (more relevance):
Result 1: RAG combines retrieval and generation for better responses.
Result 2: RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval of relevant documents with generation of responses using language models.

Results with lambda_mult=0.5 (more diversity):
Result 1: RAG combines retrieval and generation for better responses.
Result 2: The India vs Pakistan cricket match is one of the most anticipated events in the sport.


### Multiquery Retriever

In [30]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever

In [31]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [34]:
embedding_model = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    documents = all_docs,
    embedding = embedding_model
)

In [35]:
query = "How to improve energy levels and maintain balance?"

# We can solve this problem in two ways: 
# 1. Using similarity retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
results_similarity = retriever.invoke(query)

multi_query_retriever = MultiQueryRetriever.from_llm(
    llm=ChatOpenAI(model="gpt-3.5-turbo"),
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5})
)
results_multi_query = multi_query_retriever.invoke(query)

print("Results using similarity retriever:")
for i, doc in enumerate(results_similarity):
    print(f"Result {i+1} (Source: {doc.metadata['source']}): {doc.page_content}")

print("\nResults using multi-query retriever:")
for i, doc in enumerate(results_multi_query):
    print(f"Result {i+1} (Source: {doc.metadata['source']}): {doc.page_content}")

Results using similarity retriever:
Result 1 (Source: H5): Drinking sufficient water throughout the day helps maintain metabolism and energy.
Result 2 (Source: H4): Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Result 3 (Source: H1): Regular walking boosts heart health and can reduce symptoms of depression.
Result 4 (Source: H3): Deep sleep is crucial for cellular repair and emotional regulation.
Result 5 (Source: I1): The solar energy system in modern homes helps balance electricity demand.

Results using multi-query retriever:
Result 1 (Source: H5): Drinking sufficient water throughout the day helps maintain metabolism and energy.
Result 2 (Source: H4): Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Result 3 (Source: H1): Regular walking boosts heart health and can reduce symptoms of depression.
Result 4 (Source: H3): Deep sleep is crucial for cellular repair and emotional regulation.
Result 5 (Source: H2): Consuming 

### Contextual Compression Retriever

* Advanced retriever that improves retrieval quality by compressing documents after retrieval, keeping only the relevant content based on the user's query. 

In [38]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [39]:
documents = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [40]:
embedding_model = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    documents = documents,
    embedding = embedding_model
)

In [43]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

compressor = LLMChainExtractor.from_llm(ChatOpenAI(model="gpt-3.5-turbo"))

contextual_compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

query = "What is photosynthesis?"

results = contextual_compression_retriever.invoke(query)

print("Results using contextual compression retriever:")
for i, doc in enumerate(results):
    print(f"Result {i+1} (Source: {doc.metadata['source']}): {doc.page_content}")

Results using contextual compression retriever:
Result 1 (Source: Doc1): Photosynthesis is the process by which green plants convert sunlight into energy.
Result 2 (Source: Doc2): The chlorophyll in plant cells captures sunlight during photosynthesis.
